In [ ]:
import torch
import sys
import numpy as np
from scipy.optimize import minimize
import os
import tqdm as tqdm
from sklearn.metrics import accuracy_score

sys.path.append('./rsbench-code/rsseval/rss')

from datasets.halfmnist import HALFMNIST
from datasets.addmnist import ADDMNIST
from models.mnistltn import MnistLTN
from models.mnistdpl import MnistDPL
from argparse import Namespace

args = Namespace(
    backbone="conceptizer",
    preprocess=0,
    finetuning=0,
    batch_size=128,
    n_epochs=100,
    validate=1,
    dataset="addmnist",
    lr=0.001,
    exp_decay=0.99,
    warmup_steps=0,
    wandb=None,
    task="addition",
    boia_model="ce",
    model="mnistdpl",
    c_sup=0,
    which_c=[-1],
    joint=False,
    boia_ood_knowledge=True,
 )

## Utilities

In [18]:
def get_single_concept_gold_priors(hidden_labels, num_classes):
    """
    Computes the exact marginal distribution (r) from the hidden gold labels.
    
    Args:
        hidden_labels: A list or numpy array of all ground truth concept in your dataset.
        num_classes: The total number of unique classes (c).
        
    Returns:
        priors: A numpy array of shape (c,) summing to 1.
    """
    # count occ. of each class
    counts = np.bincount(hidden_labels, minlength=num_classes)
    
    # normalize for probabilities
    priors = counts / np.sum(counts)
    
    return priors

def get_concept_pair_gold_priors(hidden_label_pairs, num_classes):
    """
    Computes the exact joint distribution (r) from the hidden gold label pairs.
    
    Args:
        hidden_label_pairs: A list or numpy array of all ground truth concept pairs in your dataset.
        num_classes: The total number of unique classes (c).
        
    Returns:
        priors: A numpy array of shape (c, c) summing to 1.
    """
    # count occ. of each class pair
    counts = np.zeros((num_classes, num_classes))
    for pair in hidden_label_pairs:
        counts[pair[0], pair[1]] += 1
    
    # normalize for probabilities
    priors = counts / np.sum(counts)
    
    return priors

def get_sigma_matrix(num_classes, priors, symbolic_function):
    """
    Constructs the sigma matrix (Σ_σ,r) described in Equation (1) which depends on the class priors.
    """
    dim = num_classes ** 2
    Sigma = np.zeros((dim, dim))
    
    # iterate over all possible gold label pairs (i, j)
    for i in range(num_classes):
        for j in range(num_classes):
            # probabilty of this gold label pair occurring (assuming independence) i.e. P(Y=i) * P(Y=j)
            prob_gold = priors[i] * priors[j]
            
            # the correct weak label for this gold label pair
            s_true = symbolic_function(i, j)
            
            for i_prime in range(num_classes):
                for j_prime in range(num_classes):
                    s_pred = symbolic_function(i_prime, j_prime) # s' = σ(i', j')
                    
                    # if weak labels differ, this contributes to the Partial Risk
                    if s_true != s_pred:
                        # u corresponds to confusion H[i, i_prime]
                        # v corresponds to confusion H[j, j_prime]
                        u = i * num_classes + i_prime
                        v = j * num_classes + j_prime
                        
                        Sigma[u, v] += prob_gold        
    return Sigma

def compute_class_specific_risk_bound(
    target_class_index,
    observed_partial_risk,
    num_classes,
    Sigma
):
    """
    Solves optimization program (2) to find the worst-case risk bound R_j(f).
    """
    dim = num_classes ** 2
    
    # Objective: Maximize risk R_j(f)
    # since scipy only does minimization, we minimize the 'Probability Correct' instead i.e.
    # Risk = 1 - Probability Correct
    def objective(h):
        # the diagonal entry H[j, j] corresponds to P(Pred=j | Gold=j)
        # in the flattended vector h, this is at index (j * num_classes + j)
        correct_pred_idx = target_class_index * num_classes + target_class_index
        prob_correct = h[correct_pred_idx]
        
        return prob_correct

    # Constraint 1: Partial Risk Match (Eq 2, first constraint)
    # h^T * Sigma * h = R_P    # Constraint 1: Partial Risk Match (Eq 2, first constraint)
    # h^T * Sigma * h = R_P
    def constraint_partial_risk(h):
        return (h.T @ Sigma ) @ h - observed_partial_risk
    
    # Constraint 2: Normalization (Eq 2, third constraint)
    # Each row of H must sum to 1
    constraints = [{'type': 'eq', 'fun': constraint_partial_risk}]
    
    for r in range(num_classes):
        def row_sum_constraint(h, row_idx=r):
            start = row_idx * num_classes
            end = start + num_classes
            return np.sum(h[start:end]) - 1.0
        constraints.append({'type': 'eq', 'fun': row_sum_constraint})

    # Bounds: Probabilities must be between 0 and 1 (Eq 2, second constraint)
    bounds = [(0, 1) for _ in range(dim)]
    
    # Initial guess: Identity matrix
    h0 = np.eye(num_classes).flatten()
    
    # Optimization
    result = minimize(
        objective, 
        h0, 
        method='SLSQP', 
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 1000, 'ftol': 1e-6}
    )
    
    if not result.success:
        print(f"Warning: Optimization failed for class {target_class_index}")
        return None
        
    # Convert minimized Prob_Correct back to Risk
    return 1.0 - result.fun
 
def get_concepts_and_labels_mnist(
    out_labels, out_concepts, true_concepts, dataset_name, is_ood=False
):
    label_logits = out_labels
    
    if not is_ood and dataset_name.lower() in ["shortmnist", "clipshortmnist"]:
        label_logits = label_logits.clone()
        allowed = torch.tensor([6, 10, 12], device=label_logits.device)
        disallowed = torch.ones(label_logits.size(1), dtype=torch.bool, device=label_logits.device)
        disallowed[allowed] = False
        label_logits[:, disallowed] = 0

    predicted_labels = torch.argmax(label_logits, dim=-1)
    predicted_concepts = torch.argmax(out_concepts, dim=-1)

    predicted_concepts = predicted_concepts.reshape(-1) # from [batch_size, 2] to [batch_size * 2]
    refactored_true_concepts = true_concepts.reshape(-1) # from [batch_size, 2] to [batch_size * 2]

    return predicted_labels, predicted_concepts, refactored_true_concepts 

## Retrieve the Observed Partial Risk on the validation set.

In [19]:
# sigma
def symbolic_function(y1, y2):
    return y1 + y2

# load dataset
if args.dataset.lower() == "halfmnist":
    dataset = HALFMNIST(args=args)
elif args.dataset.lower() == "addmnist":
    dataset = ADDMNIST(args=args)

encoder, _ = dataset.get_backbone()
n_images, c_split = dataset.get_split()

# load model
if args.model.lower() == "mnistltn":
    model = MnistLTN(encoder, n_images=n_images, c_split=c_split, args=args)
elif args.model.lower() == "mnistdpl":
    model = MnistDPL(encoder, n_images=n_images, c_split=c_split, args=args)
    
model.device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(model.device)

if hasattr(model, "encoder"):
    model.encoder.to(model.device)
if hasattr(model, "net"):
    model.net.to(model.device)
    

seed = 0
model_path = f"./rsbench-code/rsseval/rss/best_model_{args.dataset}_{args.model}"    
current_model_path = f"{model_path}_{seed}.pth"

if not os.path.exists(current_model_path):
    print(f"{current_model_path} is missing...")
else:
    print(f"Loading {current_model_path}...")

try:
    # retrieve the status dict
    model_state_dict = torch.load(current_model_path)
    # Load the model status dict
    model.load_state_dict(model_state_dict)
except Exception as e:
    print(e)
    
model.eval()

train_loader, val_loader, test_loader = dataset.get_data_loaders()

true_labels = []
predicted_labels = []
true_concepts = []
true_concept_pairs = [] # pairs of (c1, c2)

for i, data in enumerate(tqdm.tqdm(val_loader)):
    images, labels, concepts = data
    images, labels, concepts = (
        images.to(model.device),
        labels.to(model.device),
        concepts.to(model.device),
    )
    
    out_dict = model(images)
    
    out_label, out_concept, concepts_flattened = get_concepts_and_labels_mnist(
        out_dict["YS"], out_dict["pCS"], concepts, args.dataset, is_ood=False
    )
    
    true_labels.append(labels.cpu().numpy())
    predicted_labels.append(out_label.detach().cpu().numpy())
    true_concepts.append(concepts_flattened.cpu().numpy())
    true_concept_pairs.append(concepts.cpu().numpy()) # (c1, c2)

true_labels = np.concatenate(true_labels, axis=0)
predicted_labels = np.concatenate(predicted_labels, axis=0)
true_concepts = np.concatenate(true_concepts, axis=0)
true_concept_pairs = np.concatenate(true_concept_pairs, axis=0)

# print(f"True labels shape {true_labels.shape}.")
# print(f"Predicted labels shape {predicted_labels.shape}.")
# print(f"True concepts shape {true_concepts.shape}.")
# print(f"True concept pairs shape {true_concept_pairs.shape}.")
label_accuracy = accuracy_score(true_labels, predicted_labels)
observed_partial_risk = 1 - label_accuracy

# unique_classes = np.unique(true_labels) # [0, 1, 5, 6] in half-MNIST
# unique_classes
print(f"Observed Partial Risk (R_P) on validation set: {observed_partial_risk}")

Loading ./rsbench-code/rsseval/rss/best_model_addmnist_mnistdpl_0.pth...
Loading train data
Loading val data
Loading test data


100%|██████████| 94/94 [00:01<00:00, 62.40it/s]

Observed Partial Risk (R_P) on validation set: 0.9299999999999999


# Optimization

In [ ]:
C = 5 if args.dataset == "halfmnist" else 10  # Number of classes in HalfMNIST
# Compute Gold Priors (r) directly from ground truth
gold_priors = get_single_concept_gold_priors(true_concepts, C)
# gold_priors = get_concept_pair_gold_priors(true_concept_pairs, C)

# Compute Sigma Matrix using Gold Priors
Sigma = get_sigma_matrix(C, gold_priors, symbolic_function)

class_specific_risks = []

# Compute Risk Bound for each class
for target_class in range(C):
    risk_bound = compute_class_specific_risk_bound(
        target_class, 
        observed_partial_risk, 
        C, 
        Sigma
    )
    
    class_specific_risks.append(risk_bound)

    print(f"Upper bound risk for Class {target_class}: {risk_bound}")
    
print("Class-Specific Risk Bounds:", class_specific_risks)

Upper bound risk for Class 0: 0.9999999999999974
Upper bound risk for Class 1: 1.0
Upper bound risk for Class 2: 1.0
Upper bound risk for Class 3: 0.9999999999999968
Upper bound risk for Class 4: 1.0
Upper bound risk for Class 5: 0.9999999999999997
Upper bound risk for Class 6: 1.0
Upper bound risk for Class 7: 0.9999999999999998
Upper bound risk for Class 8: 0.9999999999999987
Upper bound risk for Class 9: 0.9999999999999977
Class-Specific Risk Bounds: [0.9999999999999974, 1.0, 1.0, 0.9999999999999968, 1.0, 0.9999999999999997, 1.0, 0.9999999999999998, 0.9999999999999987, 0.9999999999999977]
